# Figura 5: Curva de Patrimônio com Drawdown Máximo

Visualização de curva de patrimônio acumulada com máximo drawdown (MDD) destacado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Simulação de retornos diários
np.random.seed(42)
n_days = 250
daily_returns = np.random.normal(0.0005, 0.015, n_days)

# Injetar alguns drawdowns
daily_returns[50:60] = np.random.normal(-0.01, 0.008, 10)
daily_returns[120:135] = np.random.normal(-0.008, 0.006, 15)
daily_returns[200:210] = np.random.normal(-0.012, 0.010, 10)

# Calcular patrimônio acumulado
initial_capital = 100000
equity = initial_capital * np.cumprod(1 + daily_returns)

# Calcular running maximum
running_max = np.maximum.accumulate(equity)

# Calcular drawdown
drawdown = (equity - running_max) / running_max

# Encontrar MDD
mdd_idx = np.argmin(drawdown)
mdd_value = drawdown[mdd_idx]

# Encontrar o pico anterior ao MDD
peak_idx = np.argmax(equity[:mdd_idx+1])

# Criar figura
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Subplot 1: Curva de patrimônio
days = np.arange(n_days)
axes[0].plot(days, equity, 'b-', linewidth=2, label='Patrimônio acumulado')
axes[0].plot(days, running_max, 'g--', linewidth=1.5, alpha=0.7, label='Máximo acumulado')

# Destacar período de MDD
axes[0].fill_between(days[peak_idx:mdd_idx+1], equity[peak_idx:mdd_idx+1], 
                      running_max[peak_idx:mdd_idx+1], alpha=0.3, color='red',
                      label=f'MDD: {mdd_value*100:.1f}%')
axes[0].plot([peak_idx, mdd_idx], [equity[peak_idx], equity[mdd_idx]], 'ro-', 
             markersize=8, linewidth=2)

# Anotações
axes[0].annotate(f'Pico: R$ {equity[peak_idx]:,.0f}', 
                 xy=(peak_idx, equity[peak_idx]), xytext=(peak_idx+20, equity[peak_idx]+5000),
                 fontsize=10, weight='bold',
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
axes[0].annotate(f'Vale: R$ {equity[mdd_idx]:,.0f}', 
                 xy=(mdd_idx, equity[mdd_idx]), xytext=(mdd_idx+20, equity[mdd_idx]-5000),
                 fontsize=10, weight='bold',
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

axes[0].set_ylabel('Patrimônio (R$)', fontsize=12, weight='bold')
axes[0].set_title('Curva de Patrimônio com Máximo Drawdown (MDD)', fontsize=13, weight='bold')
axes[0].legend(loc='upper left', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R$ {x/1000:.0f}k'))

# Subplot 2: Drawdown ao longo do tempo
axes[1].fill_between(days, drawdown*100, 0, alpha=0.4, color='red', label='Drawdown')
axes[1].plot(days, drawdown*100, 'r-', linewidth=1.5)
axes[1].plot(mdd_idx, mdd_value*100, 'ro', markersize=10, label=f'MDD = {mdd_value*100:.1f}%')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

axes[1].set_xlabel('Dias de negociação', fontsize=12, weight='bold')
axes[1].set_ylabel('Drawdown (%)', fontsize=12, weight='bold')
axes[1].set_title('Série de Drawdown Diário', fontsize=13, weight='bold')
axes[1].legend(loc='lower left', fontsize=11)
axes[1].grid(True, alpha=0.3)

# Texto informativo
stats_text = f'Estatísticas de Performance:\n'
stats_text += f'Retorno total: {(equity[-1]/initial_capital - 1)*100:.1f}%\n'
stats_text += f'MDD: {mdd_value*100:.1f}%\n'
stats_text += f'Duração MDD: {mdd_idx - peak_idx} dias'
axes[1].text(0.98, 0.35, stats_text, transform=axes[1].transAxes,
             fontsize=10, verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('/Users/fteodoro/Dropbox/Doutorado/Tese/figuras/fig_05_equity_mdd.pdf', dpi=300, bbox_inches='tight')
plt.savefig('/Users/fteodoro/Dropbox/Doutorado/Tese/figuras/fig_05_equity_mdd.png', dpi=300, bbox_inches='tight')
plt.show()

print('Figura salva: fig_05_equity_mdd.{pdf,png}')